#RuRoberta

In [ ]:
import math
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import precision_recall_fscore_support

In [ ]:
MODEL_NAME = "ai-forever/ruRoberta-large"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie roberta.embeddings.word_embeddings.weight to lm_head.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
RobertaForMaskedLM LOAD REPORT from: ai-forever/ruRoberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
raw_datasets = load_dataset(
    "text",
    data_files={
        "train": "/content/drive/MyDrive/datasets/train_mlm.txt",
        "validation": "/content/drive/MyDrive/datasets/test_mlm.txt"
    }
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples["text"])

tokenized = raw_datasets.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/469 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

In [ ]:
BLOCK_SIZE = 512

def group_texts(examples):
    # Склеиваем все input_ids
    concatenated = {k: sum(examples[k], []) for k in examples}

    total_length = len(concatenated["input_ids"])
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE

    result = {
        k: [
            t[i : i + BLOCK_SIZE]
            for i in range(0, total_length, BLOCK_SIZE)
        ]
        for k, t in concatenated.items()
    }
    return result

In [ ]:
lm_datasets = tokenized.map(
    group_texts,
    batched=True
)

Map:   0%|          | 0/469 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    # маска для MLM-токенов
    mask = labels != -100

    y_true = labels[mask]
    y_pred = predictions[mask]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./ruroberta_mlm",

    eval_strategy="epoch",
    save_strategy="epoch",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,

    logging_steps=100,
    fp16=False,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,1.477865,0.682432,0.682432,0.682432
2,No log,1.214822,0.723150,0.723150,0.723150
3,No log,1.229062,0.728111,0.728111,0.728111


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=39, training_loss=1.4023602803548176, metrics={'train_runtime': 502.5658, 'train_samples_per_second': 0.149, 'train_steps_per_second': 0.078, 'total_flos': 81777015244800.0, 'train_loss': 1.4023602803548176, 'epoch': 3.0})

In [ ]:
trainer.save_model("./ruroberta_mlm")
tokenizer.save_pretrained("./ruroberta_mlm")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ruroberta_mlm/tokenizer_config.json', './ruroberta_mlm/tokenizer.json')

In [ ]:
import json
from transformers import pipeline

with open('/content/drive/MyDrive/datasets/mlm_pl.json', 'r', encoding='utf-8') as f:
    pl_mask = json.load(f)

fill_mask = pipeline(
    "fill-mask",
    model="./ruroberta_mlm",
    tokenizer="./ruroberta_mlm"
)

result = fill_mask(pl_mask)

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie roberta.embeddings.word_embeddings.weight to lm_head.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
with open('ruroberta_mlm.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=4)